In [ ]:
# Cell 1: Detect Kaggle / Colab GPU environment & install dependencies
import json, os, pathlib, sys, time, zipfile, subprocess

IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
OUT_DIR = pathlib.Path("/kaggle/working/outputs" if IS_KAGGLE else "/content/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Environment: {'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')} | Out: {OUT_DIR}")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U"] + ["git+https://github.com/huggingface/diffusers.git", 'transformers', 'accelerate', 'safetensors', 'Pillow', 'imageio[ffmpeg]', 'sentencepiece', 'protobuf'])
import torch
print("CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Cell 2: Embedded Job Specification (generated by cloud-gpu-video MCP)
JOB_SPEC = json.loads(r'''{
  "job_id": "ep01_zimage_lofi",
  "created_at": "2026-11-26 19:57:04",
  "engine": "z_image_turbo",
  "aspect_ratio": "9:16",
  "seed": 42,
  "character_lock": "2D lofi anime illustration style with crisp black outlines, featuring ONE single small minimalist white round-headed stickman character (smooth pure white circular blank head with two tiny black dot eyes and thin eyebrows, no hair, no nose) wearing a regular-proportioned dark charcoal hoodie, slim dark legs and sneakers. Normal compact human proportions, single character centered in frame, rich atmospheric background, zero text.",
  "negative_prompt": "bad anatomy, elongated body, long torso, double torso, stacked body, two heads, extra arms, extra hands, extra legs, malformed limbs, text, watermark, logo, signature, blurry, 3d render, photo",
  "voiceover_script": "Three AM. Another fifty thousand dollar account gone. You walk out into the rain wondering if you should just quit. You sit in a quiet coffee shop, open your journal, and finally see the truth. It was never your strategy. It was trading without a daily loss limit. The next morning, you cut your risk to half a percent, take one clean setup, and close the laptop. Boring discipline is what finally gets you paid. Save this if you refuse to quit.",
  "caption": "3 AM blown account -> quiet coffee shop -> 0.5% discipline -> first funded payout. Save this if you refuse to quit.",
  "hashtags": [
    "#propfirm",
    "#fundedtrader",
    "#tradingpsychology"
  ],
  "scenes": [
    {
      "scene_id": "scene_01",
      "prompt": "Medium wide shot of the single stickman sitting tired at a wooden desk at 3 AM in a dimly lit bedroom, glowing monitor showing falling red trading candlesticks casting soft crimson light, rain on window behind.",
      "duration_sec": 5.0
    },
    {
      "scene_id": "scene_02",
      "prompt": "Wide cinematic shot of the single stickman walking alone with hands in hoodie pockets along a rainy neon city sidewalk at night next to a glowing subway entrance, wet asphalt reflections.",
      "duration_sec": 5.0
    },
    {
      "scene_id": "scene_03",
      "prompt": "Medium shot of the single stickman seated on a stool at a warm wooden window counter inside a cozy dimly lit coffee shop at night, warm amber pendant lamps, steaming ceramic coffee mug, open journal, rainy neon city outside glass window.",
      "duration_sec": 5.0
    },
    {
      "scene_id": "scene_04",
      "prompt": "Medium wide shot in warm golden sunrise bedroom light, the single stickman sitting upright and calm at a clean wooden desk watching one monitor with a steady upward emerald green candlestick chart.",
      "duration_sec": 5.0
    },
    {
      "scene_id": "scene_05",
      "prompt": "Medium shot of the single stickman standing by a bright sunlit apartment window smiling gently while holding a smartphone casting a soft green glow, peaceful morning cityscape outside.",
      "duration_sec": 5.0
    }
  ]
}''')
print("Job:", JOB_SPEC["job_id"], "| Engine:", JOB_SPEC["engine"], "| Scenes:", len(JOB_SPEC["scenes"]))


In [ ]:
# Cell 3: Load GPU Pipeline & Generate All Scenes
import torch
from diffusers import ZImagePipeline
pipe = ZImagePipeline.from_pretrained("Tongyi-MAI/Z-Image-Turbo", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
pipe.enable_sequential_cpu_offload()
NUM_STEPS = 9
GUIDANCE = 0.0
W, H = (720, 1280) if JOB_SPEC.get("aspect_ratio", "9:16") == "9:16" else (1280, 720)

CHAR_LOCK = JOB_SPEC.get("character_lock", "")
NEG = JOB_SPEC.get("negative_prompt", "text, watermark, signature, blurry, deformed, extra limbs, 3d render")
BASE_SEED = int(JOB_SPEC.get("seed", 42))
manifest = {"job_id": JOB_SPEC["job_id"], "engine": JOB_SPEC["engine"], "files": []}

for idx, sc in enumerate(JOB_SPEC["scenes"]):
    sid = sc.get("scene_id") or f"scene_{idx + 1:02d}"
    full_prompt = f"{CHAR_LOCK} {sc['prompt']}".strip()
    gen = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(BASE_SEED + idx)
    t0 = time.time()
    img = pipe(prompt=full_prompt, width=W, height=H, num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE, generator=gen).images[0]
    out_path = OUT_DIR / f"{sid}.png"
    img.save(out_path)
    print(f"[OK] {sid} -> {out_path.name} ({time.time() - t0:.1f}s)")
    manifest["files"].append(out_path.name)

(OUT_DIR / "output_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
zip_path = OUT_DIR.parent / f"ep01_zimage_lofi_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob("*")):
        zf.write(p, p.name)
print("PACKED:", zip_path, "| Total files:", len(manifest["files"]))


In [ ]:
# Cell 4: Option B Auto-Sync Outputs Back to GitHub (via Colab Secret GITHUB_TOKEN) + Drive Backup
if IS_COLAB:
    _gh_synced = False
    _gh_owner, _gh_repo, _gh_branch = 'prayogayudistira6-arch', 'gpu-video-workers', 'main'
    if _gh_owner and _gh_repo:
        try:
            import base64, urllib.request, urllib.error
            from google.colab import userdata
            _tok = (userdata.get("GITHUB_TOKEN") or "").strip()
            if _tok:
                _hdr = {"Authorization": f"Bearer {_tok}", "Accept": "application/vnd.github+json", "Content-Type": "application/json"}
                for p in sorted(OUT_DIR.glob("*")):
                    if p.suffix.lower() not in (".png", ".jpg", ".jpeg", ".mp4", ".json"):
                        continue
                    _api = f"https://api.github.com/repos/{_gh_owner}/{_gh_repo}/contents/cloud_gpu_jobs/ep01_zimage_lofi/outputs/{p.name}"
                    _sha = None
                    try:
                        with urllib.request.urlopen(urllib.request.Request(_api + f"?ref={_gh_branch}", headers=_hdr)) as _r:
                            _sha = json.loads(_r.read().decode()).get("sha")
                    except Exception:
                        pass
                    _body = {"message": f"colab-worker: upload ep01_zimage_lofi/{p.name}", "content": base64.b64encode(p.read_bytes()).decode("ascii"), "branch": _gh_branch}
                    if _sha:
                        _body["sha"] = _sha
                    _req = urllib.request.Request(_api, data=json.dumps(_body).encode("utf-8"), headers=_hdr, method="PUT")
                    with urllib.request.urlopen(_req) as _pr:
                        print("[GITHUB SYNC OK]", p.name, _pr.status)
                _gh_synced = True
        except Exception as exc:
            print("GitHub auto-sync skipped (add Colab Secret GITHUB_TOKEN to enable):", exc)
    if not _gh_synced:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            drv_dir = pathlib.Path("/content/drive/MyDrive/AI_MONEY_SUITE_OUTPUTS/ep01_zimage_lofi")
            drv_dir.mkdir(parents=True, exist_ok=True)
            import shutil
            for p in sorted(OUT_DIR.glob("*")):
                shutil.copy2(p, drv_dir / p.name)
            print("SYNCED TO GOOGLE DRIVE:", drv_dir)
        except Exception as exc:
            print("Drive mount skipped (download /content/ep01_zimage_lofi_outputs.zip manually):", exc)
